# EmbdGuard GPU Test

Minimal notebook to verify GPU training works with EmbeddingBag + KJT on Kaggle T4.

In [ ]:
%%bash
echo "=== System ==="
cat /proc/driver/nvidia/version 2>/dev/null || echo "No NVIDIA driver info"
echo ""
echo "=== nvidia-smi ==="
nvidia-smi 2>/dev/null || echo "nvidia-smi not found"
echo ""
echo "=== PyTorch CUDA ==="
python -c "
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA compiled version: {torch.version.cuda}')
print(f'cuDNN version: {torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None}')
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
print(f'torch.cuda.device_count(): {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'    Capability: {torch.cuda.get_device_capability(i)}')
        print(f'    Memory: {torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GiB')
"

In [ ]:
import torch
import torch.nn as nn
import time

print(f"PyTorch {torch.__version__}")
print(f"CUDA built: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.init()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("WARNING: CUDA not available!")

print(f"\nUsing device: {device}")

## Test 1: Basic CUDA tensor ops

In [ ]:
# Basic tensor on device
x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)
z = x @ y
print(f"Tensor device: {z.device}")
print(f"Result shape: {z.shape}")
if device.type == "cuda":
    print(f"GPU memory: {torch.cuda.memory_allocated()/1024**2:.1f} MiB")

## Test 2: nn.EmbeddingBag on GPU

In [ ]:
N_USERS, N_ITEMS, EMBED_DIM = 6040, 3706, 64

user_bag = nn.EmbeddingBag(N_USERS, EMBED_DIM, mode="mean").to(device)
item_bag = nn.EmbeddingBag(N_ITEMS, EMBED_DIM, mode="mean").to(device)

print(f"user_bag weight device: {user_bag.weight.device}")
print(f"item_bag weight device: {item_bag.weight.device}")

# Forward pass
batch = 2048
user_ids = torch.randint(0, N_USERS, (batch,), device=device)
item_ids = torch.randint(0, N_ITEMS, (batch,), device=device)
offsets_u = torch.arange(batch + 1, dtype=torch.long, device=device)
offsets_i = torch.arange(batch + 1, dtype=torch.long, device=device)

u_emb = user_bag(user_ids, offsets_u[:-1])
i_emb = item_bag(item_ids, offsets_i[:-1])
print(f"u_emb: {u_emb.shape} on {u_emb.device}")
print(f"i_emb: {i_emb.shape} on {i_emb.device}")

if device.type == "cuda":
    print(f"GPU memory: {torch.cuda.memory_allocated()/1024**2:.1f} MiB")

## Test 3: Full Two-Tower training loop on GPU

In [ ]:
class SimpleTwoTower(nn.Module):
    """Minimal Two-Tower using EmbeddingBag (same pattern as EmbdGuard)."""
    def __init__(self, n_users, n_items, embed_dim, hidden):
        super().__init__()
        self.user_emb = nn.EmbeddingBag(n_users, embed_dim, mode="mean")
        self.item_emb = nn.EmbeddingBag(n_items, embed_dim, mode="mean")
        self.user_mlp = nn.Sequential(nn.Linear(embed_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.item_mlp = nn.Sequential(nn.Linear(embed_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, user_ids, item_ids, offsets, labels):
        n = len(user_ids)
        u = self.user_mlp(self.user_emb(user_ids, offsets))
        i = self.item_mlp(self.item_emb(item_ids, offsets))
        u = nn.functional.normalize(u, dim=-1)
        i = nn.functional.normalize(i, dim=-1)
        logits = (u * i).sum(dim=1) * 10.0
        return self.loss_fn(logits, labels)

model = SimpleTwoTower(N_USERS, N_ITEMS, EMBED_DIM, 64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Model device: {next(model.parameters()).device}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop — 50 steps, simulating MovieLens scale
BATCH = 4096
N_STEPS = 50

model.train()
start_time = time.time()

for step in range(N_STEPS):
    user_ids = torch.randint(0, N_USERS, (BATCH,), device=device)
    item_ids = torch.randint(0, N_ITEMS, (BATCH,), device=device)
    offsets = torch.arange(BATCH + 1, dtype=torch.long, device=device)
    labels = torch.cat([torch.ones(BATCH // 2), torch.zeros(BATCH // 2)]).to(device)

    loss = model(user_ids, item_ids, offsets[:BATCH], labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"  Step {step}: loss={loss.item():.4f}")

elapsed = time.time() - start_time
print(f"\n{N_STEPS} steps in {elapsed:.2f}s ({N_STEPS/elapsed:.1f} steps/sec)")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU memory: {torch.cuda.memory_allocated()/1024**2:.1f} MiB")
    print(f"GPU max memory: {torch.cuda.max_memory_allocated()/1024**2:.1f} MiB")

## Test 4: EmbdGuard's pure-PyTorch EBC + KJT fallback on GPU

In [ ]:
%%bash
if [ ! -d "embdguard" ]; then
    git clone https://github.com/aliafzal/embdguard.git
fi

In [ ]:
import sys
REPO = os.path.abspath("embdguard")
DL_DIR = os.path.join(REPO, "dlattack_research")
sys.path.insert(0, DL_DIR)

from src.model import build_ebc, TwoTower, TwoTowerTrainTask, make_kjt, make_optimizer
from src.train import _negative_sample_tensors

print(f"Using device: {device}")

# Build model on device
ebc = build_ebc(N_USERS, N_ITEMS, EMBED_DIM, device=device)
tt = TwoTower(ebc, layer_sizes=[128, 64], device=device)
model2 = TwoTowerTrainTask(tt)
model2.to(device)
opt2 = make_optimizer(model2)

print(f"EBC weight device: {ebc.embedding_bags['t_user_id'].weight.device}")
print(f"MLP weight device: {tt.user_proj[0].weight.device}")
print(f"Model device: {next(model2.parameters()).device}")

In [ ]:
# Training with KJT — the exact pattern used in the main notebook
BATCH = 4096
N_STEPS = 50

pos_u = torch.randint(0, N_USERS, (100000,), device=device)
pos_i = torch.randint(0, N_ITEMS, (100000,), device=device)

model2.train()
start_time = time.time()

users, items, labels = _negative_sample_tensors(pos_u, pos_i, N_ITEMS, 4, device)
print(f"Sampled tensors — users: {users.device}, items: {items.device}, labels: {labels.device}")

for step in range(N_STEPS):
    start = (step * BATCH) % (len(users) - BATCH)
    end = start + BATCH
    kjt = make_kjt(users[start:end], items[start:end])
    loss, _ = model2(kjt, labels[start:end])
    opt2.zero_grad()
    loss.backward()
    opt2.step()

    if step % 10 == 0:
        print(f"  Step {step}: loss={loss.item():.4f}")

elapsed = time.time() - start_time
print(f"\n{N_STEPS} steps in {elapsed:.2f}s ({N_STEPS/elapsed:.1f} steps/sec)")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU memory: {torch.cuda.memory_allocated()/1024**2:.1f} MiB")
    print(f"GPU max memory: {torch.cuda.max_memory_allocated()/1024**2:.1f} MiB")

print("\nAll tests passed!")